# 03 - Core interpretability

Logit lens, transcription control, directional-bias analysis, and the Qwen font
sweep. Qwen2.5-VL-7B-Instruct on BoolQ.

Produces:
- `results/logit_lens/` - v1 curves, transcription scores (superseded by nb 07 for the lens)
- `results/sets/` - n=2000 failure/control sets, bias stats, logit shifts
- `results/dpi/` - font sweep at 5/8/11pt

> Reconstructed from session transcripts; outputs not embedded.

## Setup

Clone the repo, install deps, load Qwen2.5-VL-7B in bf16 across 2x T4.

**Check Accelerator = GPU T4 x2 before running.** A Kaggle batch job inherits
`None` silently and runs at ~1200 s/item on CPU. The assert below catches it.

In [ ]:
from kaggle_secrets import UserSecretsClient
token = UserSecretsClient().get_secret("GH_TOKEN")
import os, sys
if not os.path.isdir("/kaggle/working/algoverse"):
    !git clone https://{token}@github.com/bryantran21/algoverse.git /kaggle/working/algoverse
sys.path.insert(0, "/kaggle/working/algoverse"); os.chdir("/kaggle/working/algoverse")
!git config user.email "bryantran21@gmail.com"
!git config user.name  "bryantran21"

import subprocess
subprocess.run([sys.executable,'-m','pip','install','-q','transformers>=4.49.0',
    'accelerate>=0.34.0','datasets','qwen-vl-utils','typst','Pillow',
    'scikit-learn','matplotlib','tqdm'], check=True)

import torch
print("CUDA:", torch.cuda.is_available(), torch.cuda.device_count(), "devices")
assert torch.cuda.is_available(), "NO GPU - set Accelerator to T4 x2 before running"

import config
config.DEVICE_MAP = "auto"        # 2-GPU full precision
config.LOAD_IN_4BIT = False       # 4-bit perturbs activations - never use for interp
from src.inference import load_vl_model
model, processor = load_vl_model()
print("READY", flush=True)

## Build the failure and control sets

Run every item twice - once as text tokens, once as a 5pt Typst render - and
bucket by outcome.

- **failure** = text-correct AND image-wrong. The text-correct filter attributes
  failure to rendering rather than to question difficulty.
- **control** = correct in both modes. This is what makes failure-set curves
  interpretable, since the failure set is *selected* on image-wrong.

N=500 gives 62/369 and reproduces exactly across sessions; N=2000 gives 253/1450
and is what the paper uses.

In [ ]:
import config, torch, gc, numpy as np, pickle, os
gc.collect(); torch.cuda.empty_cache()

config.BATCH_SIZE = 2
config.RENDER["font_size_pt"] = 5.0        # locked operating point

from src.inference import load_items, run_inference
N = 2000
items = load_items(n=N)

txt = {r["item_id"]: r for r in run_inference(items, "text",  model=model, processor=processor)}
img = {r["item_id"]: r for r in run_inference(items, "image", model=model, processor=processor)}
by_id = {it["item_id"]: it for it in items}

fail_all = [by_id[i] for i in txt if txt[i]["correct"]==1 and img[i]["correct"]==0]
ctrl_all = [by_id[i] for i in txt if txt[i]["correct"]==1 and img[i]["correct"]==1]

f_yes = [it for it in fail_all if it["gold"]=="yes"]
f_no  = [it for it in fail_all if it["gold"]=="no"]
print(f"N={N}  failures {len(fail_all)} (yes {len(f_yes)} / no {len(f_no)})  controls {len(ctrl_all)}")

os.makedirs("results/sets", exist_ok=True)
with open("results/sets/sets_n2000.pkl","wb") as f:
    pickle.dump({"fail_all":fail_all, "ctrl_all":ctrl_all}, f)

## Logit lens (v1)

Decode the residual stream at every layer through the final norm and unembedding,
and read off P(correct answer).

**Superseded by notebook 07.** This version reconstructs from float32-stored
hidden states and disagrees with the model's own logits by up to ~1.5 units. Kept
for provenance; use `results/logit_lens_v2/` for the reported figure.

Note the device handling: with `device_map="auto"`, `lm_head` and the final norm
may live on different GPUs than `model.device` reports.

In [ ]:
import numpy as np, torch, matplotlib.pyplot as plt
from src.interp_logit_lens import run_logit_lens, plot_logit_lens
os.makedirs("results/logit_lens", exist_ok=True)

# 62/369 operating point for the v1 figure
fail, ctrl = fail_all[:62], ctrl_all[:62]
txt_curves, img_curves, ctrl_curves = run_logit_lens(model, processor, fail, ctrl)
plot_logit_lens(txt_curves, img_curves, ctrl_curves)

## Transcription control

Same images, different task: ask the model to transcribe the render verbatim and
score character-level similarity against the source passage.

This separates *perception* from *comprehension* without touching internals. It
is also the diagnostic that later caught the LLaVA result - without it, symmetric
degradation in a second model reads as evidence against the bias hypothesis
rather than as a model that cannot see.

In [ ]:
import numpy as np, torch
from src.inference import _messages_for, _flatten_images
from src.scoring import transcription_similarity, gold_text

@torch.no_grad()
def transcribe(item):
    _, images = _messages_for(item, "image")
    imgs = _flatten_images([images])
    msgs = [{"role": "user", "content":
             [{"type": "image", "image": im} for im in imgs] +
             [{"type": "text", "text": "Transcribe all text in this image exactly. Output only the transcribed text."}]}]
    text = processor.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inputs = processor(text=[text], images=imgs, return_tensors="pt").to(model.device)
    out = model.generate(**inputs, max_new_tokens=768, do_sample=False)
    return processor.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True).strip()

# Character similarity via src.scoring (SequenceMatcher, autojunk=False). The
# difflib default autojunk=True collapses correct transcriptions of >200-char
# passages toward zero and badly understates legibility -- never use it here.
scores, preds = [], []
for it in fail:
    pred = transcribe(it)
    preds.append(pred)
    scores.append(transcription_similarity(pred, gold_text(it)))
    torch.cuda.empty_cache()

scores = np.array(scores)
np.savez("results/logit_lens/transcription_scores.npz",
         scores=scores, preds=np.array(preds, dtype=object))
print(f"n={len(scores)}  mean={scores.mean():.3f}  median={np.median(scores):.3f}")
print(f"frac >0.90: {(scores>0.90).mean():.2f}   frac <0.50: {(scores<0.50).mean():.2f}")
print("\nlowest example:\n", preds[int(scores.argmin())][:300])

## Directional bias

Per-class accuracy under rendering, and the composition of image-mode errors.

The key comparison is not overall accuracy but the *asymmetry*: information loss
predicts both classes degrade, a bias predicts one class degrades while the other
may improve.

In [ ]:
import numpy as np
from scipy import stats

rows = []
for gold in ("yes", "no"):
    ids = [i for i in txt if by_id[i]["gold"] == gold]
    t = np.mean([txt[i]["correct"] for i in ids])
    m = np.mean([img[i]["correct"] for i in ids])
    rows.append((gold, len(ids), t, m, t - m))
    print(f"gold={gold:3s} n={len(ids):4d}  text {t:.3f}  image {m:.3f}  drop {t-m:.3f}")

errs = [i for i in img if img[i]["correct"] == 0]
said_no = sum(1 for i in errs if by_id[i]["gold"] == "yes")
print(f"\nimage-mode errors: {len(errs)}   gold=yes (model said no): {said_no} ({said_no/len(errs):.1%})")

f_tab = [len(f_yes), len(f_no)]
c_tab = [sum(1 for it in ctrl_all if it["gold"]=="yes"), sum(1 for it in ctrl_all if it["gold"]=="no")]
chi2, p, _, _ = stats.chi2_contingency([f_tab, c_tab])
print(f"failure vs control class skew: chi2={chi2:.1f}  p={p:.2e}")

np.savez("results/sets/bias_stats.npz",
         per_class=np.array([[r[2], r[3]] for r in rows]),
         fail_tab=np.array(f_tab), ctrl_tab=np.array(c_tab), chi2=chi2, p=p)

## Final-layer logit difference

Define `d = logit(Yes) - logit(No)` at the answer position, and measure the paired
shift `d_image - d_text` on the same items.

The prediction differs between hypotheses: information loss drives `d` *toward
zero* (the model becomes uncertain); a bias term drives it *past* zero (the model
becomes confidently wrong).

Run on both sets. Control items are correct in both modes, so any shift there is
bias unconfounded by errors.

**Save under distinct filenames.** Reusing `d_txt`/`d_img` across the two runs
once overwrote the control arrays with failure data.

In [ ]:
import numpy as np, torch
from src.activations import capture_activations

lm_head    = model.get_output_embeddings()
head_dtype = next(lm_head.parameters()).dtype
final_norm = dict(model.named_modules())["model.language_model.norm"]
norm_dev   = next(final_norm.parameters()).device
head_dev   = next(lm_head.parameters()).device
YES, NO = 9454, 2753

@torch.no_grad()
def final_logit_diff(item, mode):
    rec = capture_activations(item, mode, model, processor, save=False)
    v = torch.tensor(rec["hidden"])[-1].to(norm_dev, dtype=head_dtype)
    lg = lm_head(final_norm(v).to(head_dev)).float()
    return (lg[YES] - lg[NO]).item()

from scipy import stats
for tag, subset in (("ctrl", ctrl_all[:150]), ("fail", fail_all[:150])):
    d_txt = np.array([final_logit_diff(it, "text")  for it in subset])
    d_img = np.array([final_logit_diff(it, "image") for it in subset])
    shift = d_img - d_txt
    np.savez(f"results/sets/logit_shift_{tag}.npz",
             d_txt=d_txt, d_img=d_img, ids=[it["item_id"] for it in subset])
    t, p = stats.ttest_rel(d_img, d_txt)
    print(f"{tag}: text {d_txt.mean():+.3f}  image {d_img.mean():+.3f}  "
          f"shift {shift.mean():+.3f} (sd {shift.std():.3f}, "
          f"frac neg {(shift<0).mean():.2f})  paired t={t:.2f} p={p:.2e}")
    torch.cuda.empty_cache()

## Font sweep

Repeat the per-class accuracy and shift measurement at 5 / 8 / 11pt to test
dose-response.

Text-mode baseline is font-independent, so compute it once. Checkpoint after each
font - a run that only writes at the end loses everything on interruption.

In [ ]:
import pickle, os, numpy as np
os.makedirs("results/dpi", exist_ok=True)
config.BATCH_SIZE = 2

items_sweep = load_items(n=500)
by_id_s = {it["item_id"]: it for it in items_sweep}
txt_res  = {r["item_id"]: r for r in run_inference(items_sweep, "text", model=model, processor=processor)}
txt_diff = {}
for it in items_sweep:
    txt_diff[it["item_id"]] = final_logit_diff(it, "text")
    torch.cuda.empty_cache()
print("text baseline acc =", round(float(np.mean([txt_res[i]["correct"] for i in txt_res])), 3))

FONTS, N_SHIFT, sweep = [5.0, 8.0, 11.0], 75, {}
for fs in FONTS:
    config.RENDER["font_size_pt"] = fs
    img_res = {r["item_id"]: r for r in run_inference(items_sweep, "image",
                                                     model=model, processor=processor)}
    torch.cuda.empty_cache()
    acc_yes = np.mean([img_res[i]["correct"] for i in img_res if by_id_s[i]["gold"]=="yes"])
    acc_no  = np.mean([img_res[i]["correct"] for i in img_res if by_id_s[i]["gold"]=="no"])
    acc_all = np.mean([img_res[i]["correct"] for i in img_res])
    errs = [i for i in img_res if img_res[i]["correct"]==0]
    false_no = np.mean([by_id_s[i]["gold"]=="yes" for i in errs]) if errs else float("nan")

    ctrl_ids = [i for i in img_res if txt_res[i]["correct"]==1 and img_res[i]["correct"]==1][:N_SHIFT]
    shift = []
    for i in ctrl_ids:
        shift.append(final_logit_diff(by_id_s[i], "image") - txt_diff[i])
        torch.cuda.empty_cache()
    shift = np.array(shift)

    sweep[fs] = dict(acc_all=float(acc_all), acc_yes=float(acc_yes), acc_no=float(acc_no),
                     false_no=float(false_no), shift_mean=float(shift.mean()),
                     shift_sd=float(shift.std()), n_shift=len(shift))
    print(f"{fs:5.1f}pt  acc {acc_all:.3f}  yes {acc_yes:.3f}  no {acc_no:.3f}  "
          f"false-no {false_no:.2f}  shift {shift.mean():+.3f}", flush=True)
    with open("results/dpi/font_sweep.pkl","wb") as f:
        pickle.dump({"sweep":sweep, "N":500}, f)

config.RENDER["font_size_pt"] = 5.0

### Font sweep figure

Error bars are 95% CIs from the per-item sd. Without them the right panel implies
a dip at 8pt that the data does not support - 5pt and 8pt overlap almost entirely.

In [ ]:
import pickle, numpy as np, matplotlib.pyplot as plt
sweep = pickle.load(open("results/dpi/font_sweep.pkl","rb"))["sweep"]
fs_list = sorted(sweep)

fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))
ax[0].plot(fs_list, [sweep[f]["acc_yes"] for f in fs_list], "o-", color="#1E2761", label="gold = yes")
ax[0].plot(fs_list, [sweep[f]["acc_no"]  for f in fs_list], "s-", color="#C0392B", label="gold = no")
ax[0].set_xlabel("font size (pt)"); ax[0].set_ylabel("image-mode accuracy")
ax[0].set_title("Per-class accuracy vs render size"); ax[0].legend(); ax[0].grid(alpha=0.3)

means = [sweep[f]["shift_mean"] for f in fs_list]
yerr  = [1.96*sweep[f]["shift_sd"]/np.sqrt(sweep[f]["n_shift"]) for f in fs_list]
ax[1].errorbar(fs_list, means, yerr=yerr, fmt="^-", color="#1E2761", capsize=4)
ax[1].axhline(0, color="gray", ls=":")
ax[1].set_xlabel("font size (pt)"); ax[1].set_ylabel("mean d logit (yes-no), image - text")
ax[1].set_title("Directional bias vs render size (95% CI)"); ax[1].grid(alpha=0.3)

plt.tight_layout(); plt.savefig("results/dpi/font_sweep.png", dpi=150); plt.show()

for f in fs_list:
    d = sweep[f]; se = d["shift_sd"]/np.sqrt(d["n_shift"])
    print(f"{f:5.1f}pt  shift {d['shift_mean']:+.3f}  95% CI "
          f"[{d['shift_mean']-1.96*se:+.3f}, {d['shift_mean']+1.96*se:+.3f}]")

In [ ]:
!git add -A && git commit -m "03: logit lens, transcription, bias analysis, font sweep" && git push origin master